# Check 02 — Policy Middleware

In [ ]:
import pathlib
import sys

_root = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(_root))

from src.policies.middleware import DeterministicFirstPolicyMiddleware
from src.schemas.tool_io import (
    ExecutionMetadata,
    RiskTier,
    ToolAudit,
    ToolCallContext,
    ToolExecutionMode,
    ToolResult,
    ToolStatus,
)

In [ ]:
policy = DeterministicFirstPolicyMiddleware()

call = ToolCallContext(
    schema_version="1.0",
    call_id="call_policy_nb",
    session_id="sess_policy_nb",
    run_id="run_policy_nb",
    job_id="job_policy_nb",
    task_id="task_policy_nb",
    agent_id="agent_policy_nb",
    provider_id="openai",
    tool_name="safe_tool",
    arguments={"value": 1},
    risk_tier=RiskTier.LOW,
    is_state_changing=False,
)
decision = policy.before_tool_call(call)
assert decision.decision.value in {"allow", "deny", "escalate"}
print("before_tool_call:", decision.decision.value, decision.reason_code)

bad_result = ToolResult(
    schema_version="1.0",
    call_id="call_policy_nb",
    tool_name="safe_tool",
    status=ToolStatus.SUCCESS,
    result=None,  # triggers post-check failure for success payload missing
    execution=ExecutionMetadata(mode_used=ToolExecutionMode.DETERMINISTIC),
    audit=ToolAudit(correlation_id="call_policy_nb"),
)
checked = policy.after_tool_call(bad_result)
assert checked.status == ToolStatus.ERROR
assert checked.error.code == "POLICY_POSTCHECK_FAILED"
print("after_tool_call post-check:", checked.error.code)
print("PASS: policy middleware checks")